# 47장 · 필수 검색과 선택 검색의 종료 조건

같은 질문에서도 모든 문서를 확인하는 경로와 필요할 때만 검색하는 경로는 호출 기록이 달라집니다. 각 경로의 종료 사유를 추적합니다.

운영 안내 서비스가 문서 확인 없이 답하는 경우를 줄이기 위해 필수 검색 경로를 검토합니다. 선택형 도구 경로와 같은 입력을 처리한 뒤 호출 횟수와 종료 상태를 비교합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요.


## 채택 전 후보와 기준선의 실행 경로 분리하기

46장의 결정대로 기존 그래프를 먼저 실행합니다. 같은 요청에 후보 규칙도 따로 적용해 어떤 후보 문단이 실제로 존재하는지 확인하고 두 실행의 결과를 다른 필드에 넣습니다.

이 프로젝트의 노트북은 한 폴더에서 열어 주세요. 첫 셀은 `team-project/midterm/chapter46.json`을 먼저 찾습니다. 팀 파일이 없으면 [이 장의 시작 예시](chapter47-start.json)를 노트북 옆에 내려받으세요. 출력 첫 줄에서 어느 파일을 읽었는지 확인할 수 있습니다.

In [ ]:
from pathlib import Path
import json
import re

here = Path.cwd()
project_root = here.parent if re.fullmatch(r"chapter[0-9]+", here.name) else here
team_file = project_root / "team-project/midterm/chapter46.json"
sample_file = here / "chapter47-start.json"
input_file = team_file if team_file.is_file() else sample_file
if not input_file.is_file():
    raise FileNotFoundError("이 장의 시작 예시를 내려받아 노트북 옆에 두세요.")
incoming = json.loads(input_file.read_text(encoding="utf-8"))
print("팀 기록" if input_file == team_file else "교재 예시", input_file.name)
print(incoming['runs'])

교재 예시 chapter47-start.json
[{'id': 'Q01', 'question': '과제는 어디에 내나요?', 'action': '답변', 'expected': ['p2'], 'hits': ['p2'], 'passed': True, 'candidate': ['p2']}, {'id': 'Q02', 'question': '내 점수는 몇 점인가요?', 'action': '보류', 'expected': [], 'hits': [], 'passed': True, 'candidate': []}, {'id': 'Q03', 'question': '과제 업로드가 실패했어요. 어떻게 해야 하나요?', 'action': '답변', 'expected': ['p3'], 'hits': [], 'passed': False, 'candidate': ['p3']}]


두 번째 줄은 읽은 파일의 `runs` 값입니다.

## 고정 경로와 선택 경로의 실행 기록

완성 흐름｜질문 Q01 → 검색 노드가 p2를 읽음 → 답변 노드가 p2를 인용 → 화면에 제출 절차와 출처 표시. 46장의 기본 검색 규칙을 유지해 Q03은 여전히 빈 검색으로 끝납니다. 필요한 경우에만 찾아도 되는 팀은 선택 Tool 경로를 별도로 비교할 수 있습니다.


### 셀 01 · 선택 경로의 검색 생략

같은 Q01의 고정 Node 기록과 도구 생략 진단 기록을 읽으세요.

팀이 검색 생략을 허용하면 Q01처럼 문서가 필요한 질문까지 건너뛸 수 있습니다. 반대로 반복 상한을 두면 끝까지 근거를 찾지 못한 질문은 보류해야 합니다. 두 실패 중 이 서비스가 더 신중하게 다뤄야 할 하나를 고르고 그 이유를 기록하세요.


In [1]:
required = {'search_status': 'retrieved',
 'retrieval_calls': 1,
 'generation_calls': 1,
 'decision_calls': 0,
 'stop_reason': 'completed'}
skipped = {'search_status': 'not_requested',
 'retrieval_calls': 0,
 'generation_calls': 0,
 'decision_calls': 1,
 'stop_reason': 'search_skipped'}
print(required['retrieval_calls'], required['stop_reason'])
print(skipped['retrieval_calls'], skipped['stop_reason'])

1 completed
0 search_skipped


필수 경로는 검색 1회 뒤 completed, 생략 진단은 0회 뒤 search_skipped입니다. 검색 생략은 빈 결과를 실제로 받은 경우와 다릅니다.

두 경로에 같은 질문을 넣었는데 한쪽만 search_skipped로 끝났습니다. 도구가 등록돼 있어도 경로에 따라 호출 횟수가 0일 수 있습니다. 팀에서 문서 확인을 필수로 정했다면 정상 종료 여부와 별도로 검색 호출 기록을 검사해야 합니다.


#### 검색 횟수 0을 해석하기 전

<mark>검색을 생략한 경로와 검색했지만 문서가 없던 경로는 다릅니다.</mark> 빈 출처 목록만 보면 두 경우가 같아 보입니다. 팀 시연에서는 경로 이름과 호출 횟수를 함께 남겨 어느 조건이 실행을 끝냈는지 보여 주세요.

선수 개념 · 38장 재검색 조건과 반복 종료

#### 선택 검색이 검색과 생성을 모두 건너뛰는 경로

```mermaid
flowchart TD
ndecision["검색 결정"]
  nsearch["검색"]
  ngenerate["답변 생성"]
  nskip["검색 생략"]
  nend["종료 기록"]
  ndecision --> nsearch
  nsearch --> ngenerate
  ngenerate --> nend
  ndecision --> nskip
  nskip --> nend
```

셀의 skipped 기록을 따라갑니다. 결정이 한 번 있었지만 검색 요청을 내지 않아 두 후속 작업의 호출 횟수가 0입니다.

## 반복 상한과 호출 횟수

도구가 같은 요청을 되풀이하면 추가 검색이 더 좋은 답으로 이어지지 않을 수 있습니다. 제공 사례의 repeat 정책에서는 검색 2회 뒤에 중단하며 답변 생성을 시작하지 않습니다. 팀은 재시도를 허용할 실패와 곧바로 알릴 실패를 구분해 선택하세요.


### 셀 02 · 반복의 검색·생성 횟수

제공한 실제 그래프 실행 기록에서 어느 시점에서 멈췄는지 읽습니다.


In [2]:
repeated = {'search_status': 'retrieved',
 'retrieval_calls': 2,
 'generation_calls': 0,
 'decision_calls': 3,
 'stop_reason': 'step_limit'}
print(repeated['retrieval_calls'], repeated['generation_calls'], repeated['stop_reason'])

2 0 step_limit


2, 0, step_limit입니다. 동일 질문을 끝없이 처리할 수 없도록 검색 상한을 적용한 결과입니다. 모의 판단 횟수로 외부 모델 비용을 계산하지 않습니다. 팀 그래프도 바꿨다면 Q01을 다시 보내 검색 호출 수와 종료한 노드를 기록하세요. 이 셀의 제공 기록만 읽었다면 진단한 원인까지만 적습니다.


## 검색을 한 번만 허용하는 그래프를 실제로 실행하기

팀은 인사에는 검색 없이 답하고, 제출 위치를 묻는 입력에는 검색을 한 번만 쓰기로 했습니다. 앞에서 읽은 정책 기록을 이번에는 실행 가능한 두 간선으로 옮깁니다.

### 셀 03 · 조건과 노드를 분리한 그래프 연결

제출 위치만 지원하는 작은 경로입니다. 판단 함수가 세는 값과 검색 노드에서 늘어나는 값을 어디에 두었는지 비교하세요.

### 질문에 따라 검색을 생략하는 팀 그래프

```mermaid
flowchart TD
    s["START"] --> c{"질문에 과제가 있는가?"}
    c -->|예| l["lookup · 검색 횟수 증가"]
    c -->|아니요| e["END"]
    l --> e
```

needs_guide가 lookup 또는 finish를 반환합니다. lookup 뒤에 END가 있어 이 구현은 질문 하나당 최대 한 번 검색합니다.

In [3]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class DeskState(TypedDict):
    question: str
    hits: list[str]
    calls: int

def needs_guide(state):
    return "lookup" if "과제" in state["question"] else "finish"

def lookup_submission(state):
    ids = ["course-guide-v1:p2"] if state["question"] == "과제는 어디에 내나요?" else []
    return {"hits": ids, "calls": state["calls"] + 1}

desk = StateGraph(DeskState)
desk.add_node("lookup", lookup_submission)
desk.add_conditional_edges(START, needs_guide, {"lookup": "lookup", "finish": END})
desk.add_edge("lookup", END)
desk_app = desk.compile()
event = desk_app.invoke({"question": "과제는 어디에 내나요?", "hits": [], "calls": 0})
print(event["calls"], event["hits"])

1 ['course-guide-v1:p2']


결과의 1은 검색 노드를 실제로 통과하며 증가한 횟수입니다. 고정 규칙으로 분기했고 외부 모델은 호출하지 않았습니다.

### 셀 04 · 다른 끝점에서 호출 횟수 읽기

인사를 넣고 같은 그래프가 검색 노드를 통과하는지 확인하세요.

### 인사말의 실제 끝점

```mermaid
flowchart TD
    s["START · 인사말"] --> c{"needs_guide · 과제가 있는가?"}
    c -->|finish| e["END · 검색 생략"]
```

인사말에는 과제가 없어 START에서 END로 바로 이동합니다.

In [4]:
hello = desk_app.invoke({"question": "안녕하세요", "hits": [], "calls": 0})
print(hello["calls"], hello["hits"])

0 []


0과 빈 목록이 함께 나옵니다. “과제 양식은?”처럼 분기는 통과하지만 문서를 못 찾는 입력을 넣으면 1과 빈 목록이 나옵니다.

### 항상 찾아야 하는 팀 서비스라면

검색이 필수인 규정 조회 서비스는 시작점에서 lookup으로 직접 연결할 수 있습니다. 인사와 규정 질문을 각각 보내고, 선택 경로와 검색 횟수·사용자 안내가 어떻게 달라졌는지 비교하세요.

<mark>후보가 비었다는 사실만으로는 어떤 간선을 지났는지 알 수 없습니다.</mark> 노드 진입 시에만 calls를 증가시키면 검색 생략과 검색 실패를 같은 빈 목록에서 구별할 수 있습니다.

48장 화면에는 현재 응답의 hits와 calls를 함께 전달해 빈 상태를 설명하세요.

## 팀 사례에 적용하기 · 채택 전 후보와 기준선의 실행 경로 분리하기

위 예제에 첫 셀에서 읽어 둔 팀 기록을 넣어 봅니다.

### 팀 문항이 거치는 노드

```mermaid
flowchart TD
    s["START"] --> c{"질문에 과제가 있는가?"}
    c -->|예| l["lookup · 검색 횟수 증가"]
    c -->|아니요| e["END"]
    l --> e
```

graph_ids는 그래프가 조회한 문단입니다. 뒤에서 따로 계산하는 experiment_ids는 같은 질문을 후보 규칙에 적용한 결과입니다.

In [ ]:
paths = []
rule = incoming["candidate_rule"]
for row in incoming["runs"]:
    event = desk_app.invoke({"question": row["question"], "hits": [], "calls": 0})
    baseline_ids = [key for key in event["hits"] if key.split(":")[-1] in incoming["documents"]]
    experiment_ids = list(baseline_ids)
    if event["calls"] and any(term in row["question"] for term in rule["terms"]):
        if rule["document"] in incoming["documents"]:
            experiment_ids = ["course-guide-v1:" + rule["document"]]
    paths.append({**row, "graph_calls": event["calls"], "graph_ids": baseline_ids,
                  "experiment_ids": experiment_ids})
new = {"runs": paths, "documents": incoming["documents"], "decision": incoming["decision"]}
print([(row["id"], row["graph_ids"], row["experiment_ids"]) for row in paths])

[('Q01', ['course-guide-v1:p2'], ['course-guide-v1:p2']), ('Q02', [], []), ('Q03', [], ['course-guide-v1:p3'])]


Q03에서 graph_ids는 비고 experiment_ids에는 p3이 남습니다. 이 팀은 후보를 정식으로 채택하지 않았습니다. 이후 화면 비교에서도 두 경로의 결과를 구별해야 합니다.

In [ ]:
output_file = project_root / "team-project/midterm/chapter47.json"
output_file.parent.mkdir(parents=True, exist_ok=True)
with output_file.open("a+", encoding="utf-8") as handle:
    handle.seek(0)
    previous_text = handle.read()
    if previous_text and json.loads(previous_text) != new:
        raise FileExistsError("같은 장의 다른 결과가 있습니다. 기존 JSON을 다른 이름으로 보관한 뒤 다시 실행하세요.")
    if not previous_text:
        json.dump(new, handle, ensure_ascii=False, indent=2)
        handle.write("\n")
print(output_file.relative_to(project_root).as_posix())

team-project/midterm/chapter47.json


48장 첫 셀이 이 파일을 읽습니다. 입력을 바꿔 다시 실행할 때는 기존 결과 파일의 이름을 바꿔 보관하세요.

## 돌아보기

다음 화면에 건넬 값은 답과 출처, 또는 종료 사유입니다. 이 프로젝트 예시는 요청마다 새 상태를 쓰므로 화면이 보유한 대화 기록과 체크포인터에 저장된 상태를 혼동하지 마세요.
